# Advanced document indexing

## Splitting and ingesting the content of a single URL (on Cornwall)

### Preparing the Chroma DB collections

In [1]:
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [2]:
ollama_embeddings = OllamaEmbeddings(model="bge-m3")

cornwall_granular_collection = Chroma(  # A
    collection_name="cornwall_granular",
    embedding_function=ollama_embeddings,
)

cornwall_granular_collection.reset_collection()  # B

# A Create a Chroma collection using local Ollama embeddings.
# B Reset the collection in case it already exists.

In [3]:
cornwall_coarse_collection = Chroma( # A 
    collection_name="cornwall_coarse",
    embedding_function=ollama_embeddings
)

cornwall_coarse_collection.reset_collection() # B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

### Loading the HTML content with the AsyncHtmlLoader

In [4]:
from langchain_community.document_loaders import AsyncHtmlLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()
len(docs)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.89it/s]


1

### Splitting into granular chunks with the HTMLSectionSplitter

In [6]:
from langchain_text_splitters import HTMLSectionSplitter

In [7]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(
            html_string) #B
        all_chunks.extend(temp_chunks) 

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

granular_chunks = split_docs_into_granular_chunks(docs)

# Ingesting granular chunks
cornwall_granular_collection.add_documents(documents=granular_chunks)

# Searching granular chunks
results = cornwall_granular_collection.similarity_search(query="Events or festivals in Cornwall", k=3)
for doc in results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance 

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter

In [8]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=300)

def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #A 
    coarse_chunks = text_splitter.split_documents(
        text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

coarse_chunks = split_docs_into_coarse_chunks(docs)

# Ingesting coarse chunks
cornwall_coarse_collection.add_documents(documents=coarse_chunks)

# Searching coarse chunks
results = cornwall_coarse_collection.similarity_search(query="Events or festivals in Cornwall", k=3)
for doc in results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corni

## Splitting and ingesting the content of various URLs (across UK destinations)

In [10]:
# Preparing the Chroma DB collections
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=ollama_embeddings
)

uk_granular_collection.reset_collection() #B
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=ollama_embeddings
)

uk_coarse_collection.reset_collection() #B

### Splitting and ingesting HTML content with the HTMLSectionSplitter

In [11]:
# Reduce this list if you want to save on processing fees
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #C
    docs =  html_loader.load() #D
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists 
#C Loader for one destination
#D Documents of one destination

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.88it/s]


{'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.11it/s]


{'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.27it/s]


...

{'source': 'https://en.wikivoyage.org/wiki/Rye_(England)', 'title': 'Rye (England) – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.21it/s]


{'source': 'https://en.wikivoyage.org/wiki/Seaford', 'title': 'Seaford – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.01it/s]


{'source': 'https://en.wikivoyage.org/wiki/Ashdown_Forest', 'title': 'Ashdown Forest – Travel guide at Wikivoyage', 'language': 'en'}


In [12]:
# Searching
granular_results = uk_granular_collection.similarity_search(query="Events or festivals in East Sussex", k=4)
for doc in granular_results:
    print(doc)

page_content='East Sussex' metadata={'Header 1': 'East Sussex'}
page_content='Sussex for free 
 [ edit ] 
 
 A Market during the Brighton Festival 
 There's plenty in Sussex for those who don't wish to spend plenty of cash on attractions: 
 
 
 Walking  - 3,500   km of walking paths, bridleways, scenic roads - all for free. 
 Go for a swim: Sussex has some of the cleanest beaches in the UK, with Brighton Beach renowned for its packed seafront, less well used areas, such as Eastbourne, Bexhill and Hastings still have facilities and cleanliness. 
 Brighton itself can be one big performance, the  Brighton Festival  and the  Brighton Festival Fringe , Features street performers, theatre groups, musicians, guided walks and a whole host of other great activities. 
 Town museums: Often they will charge, but some such as  Brighton Museum and Art Gallery  and Newhaven Museum are free (donations are gratefully welcomed though).' metadata={'Header 2': 'Sussex for free'}
page_content='Do 
 [ edit 

In [13]:
coarse_results = uk_coarse_collection.similarity_search(query="Events or festivals in East Sussex", k=4)
for doc in coarse_results:
    print(doc)

page_content='The usual chains of hotels are beginning to spring up.

The towns below have accommodation throughout the year:

  * **Eastbourne** This is one of England’s most famous seaside resorts. The elegant seafront is flanked by flowerbeds. Visitor attractions include parks and gardens, a thriving marina and the cliffs at nearby Beachy Head.
  * **Hastings and St Leonard’s** Popular seaside resorts, surrounded by stunning countryside. Hastings also has a picturesque old town.
  * **Lewes** is one of the county’s oldest towns. Attractions include the castle and Anne of Cleves’ house. Around Lewes there are many picturesque villages to visit.
  * **Rye and surrounding areas** With its steep cobbled streets and picture-postcard cottages, Rye is a charming town. Surrounding attractions include Camber Sands and Winchelsea.
  * **Seaford** is a quiet beach resort. A great base for exploring the South Downs and Seven Sisters Country Park.

_Individual town pages will have more informati

In [14]:
granular_results = uk_granular_collection.similarity_search(query="Beaches in Cornwall", k=4)
for doc in granular_results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='North Cornwall' metadata={'Header 1': 'North Cornwall'}
page_content='West Cornwall' metadata={'Header 1': 'West Cornwall'}
page_content='South Cornwall' metadata={'Header 1': 'South Cornwall'}


In [15]:
coarse_results = uk_coarse_collection.similarity_search(query="Beaches in Cornwall", k=4)
for doc in coarse_results:
    print(doc)

page_content='**South Cornwall** is in Cornwall. It includes much of the stunning Cornish
coast along the English Channel of the Atlantic Ocean.

## Towns and villages

[edit]

Map of South Cornwall

  * 50.26-5.0511 Truro — Cornwall's main centre hosts the Royal Cornwall Museum
  * 50.3311-4.20212 Cawsand — overlooks Plymouth Sound; Cawsand is within Mount Edgcumbe Country Park
  * 50.15-5.073 Falmouth — famous for its beaches, it is home to the world's third largest natural harbour
  * 50.334-4.6334 Fowey — the Fowey Regatta in mid-August attracts many yachts and sailing boats
  * 50.354-4.4545 Looe — a summer resort place with a monkey sanctuary, and an active fishing village
  * 50.408-4.2126 Saltash — "Gateway to Cornwall", a small town on the Cornwall side of the Tamar crossings
  * 50.338-4.7957 St Austell — largest town in the county and home to the Eden Project, the world's largest greenhouse
  * 50.3314-4.75788 Charlestown — seaside town used as filming location for the TV sh

## Embedding strategy

### Embedding child chunks with ParentDocumentRetriever

In [16]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

In [18]:
# Setting up the Parent Document retriever

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=ollama_embeddings,
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

In [19]:
# Ingesting the content into doc and vector store

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination 
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.87it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.16it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.20it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.24it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.01it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [20]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

['9cee5893-c234-4993-901d-be64caa83ecf',
 '35ca23fb-82a7-4044-834f-42a3fef89d82',
 '419fb900-e973-48b8-9ce4-2095faed2e25',
 '04658750-6ff1-4b7e-8ee2-021156b4021b',
 '12553a06-f236-4d5d-a028-86e8c25b7465',
 '12d04995-2805-4972-9ae9-34b90999df89',
 'ea856a58-02c4-4435-a627-23a9a3df58bc',
 'fb802e66-8a84-41b6-bf1b-8b32c514ac94',
 '23f30e15-f982-4c65-b8e6-ce30e72d7a89',
 'cb75f549-37a0-4b4f-8c75-0c4002077bea',
 'eb6cef83-84aa-44b7-878c-b99d07efa9b2',
 'ba7a843f-a793-4b62-a55d-54e015fa86a5',
 '3f6fa0b1-9e65-450f-8c95-284cc9395623',
 '556e7aa4-5294-4c7a-9c8f-be6a49aa15b5',
 '9872cd17-b62d-49ae-8798-2c9633993dd2',
 '973a1dac-767d-4647-851a-bd50ddb19683',
 'a7ec87c3-0e64-4e31-a3d4-28a891731960',
 '2c0a47bb-ac8f-403e-8fa7-533c4a9c74dc',
 '6f7c9feb-35e8-479f-a439-0578d195112f',
 'd202dbda-736b-458d-9e56-f1947eeed326',
 'ec4500e9-1dd5-4baa-bc84-f050d3e1b4a2',
 'eba9ed75-0175-49a6-b4b4-83bcb0815cac',
 'ec7fbbce-ca34-4583-870b-74db819d1971',
 '134a7a22-ffd7-46e0-b8ae-c25cc2e33803',
 'bd7d4f6b-c0ab-

In [21]:
# Performing a search on granular information

retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")
len(retrieved_docs)

4

In [22]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Penzance', 'title': 'Penzance – Travel guide at Wikivoyage', 'language': 'en'}, page_content='First Kernow bus T1 runs every 30 min between Penzance and Truro (1 hr 45\nmin), via St Erth, Hayle, Camborne, and Redruth. Change at Truro for Newquay,\nSt Austell and Bodmin. Reaching Plymouth and Exeter by bus is not worth the\nbother, take the train.\n\n### By car\n\n[edit]\n\nPenzance is a 5- to 6-hour drive from London via M4, M5, and A30. It\'s a long\nway and at some point you\'ll need to refuel. Don\'t be paying motorway prices,\nthere\'s supermarket petrol at (amongst others) M5 jcn 28 (Cullompton Tesco),\nA30 Bodmin (Asda, Launceston Rd Bodmin) and A30 Penzance (Tesco).\n\n### By boat\n\n[edit]\n\nA ferry plies between Penzance and the Isles of Scilly, daily from mid-March\nto October. The ferry (Scillonian III[dead link]) leaves Penzance around 9AM\nto reach the main island of St Mary\'s at noon; it returns at 4:30PM for\

In [23]:
# Comparing with direct semantic search on child chunks

child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")
print(len(child_docs_only))
child_docs_only[0]

4


Document(id='4d0df037-beb5-4c9a-a1c7-74becd56f73d', metadata={'title': 'Penzance – Travel guide at Wikivoyage', 'doc_id': '6bcb2b46-10ea-4836-89a9-10904f1263b2', 'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/Penzance'}, page_content='These "A"-buses, operated by First Kernow, are blue open-top double-deckers in\nsummer. For bus travel plus rail, a good deal is the _Ride Cornwall Ranger_\n(adult £13) described above. For bus only, buy a _Day Rider_ for £12 (child\n£6) from the Bus Station or from the driver on boarding - contactless bank\ncards accepted. Bus drivers also issue Ride Cornwall Rangers, but only for\nfull price, go to the station for concessions.\n\n## See\n\n[edit]')

### Embedding child chunks with MultiVectorRetriever

In [24]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

In [25]:
ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

In [28]:
# Setting up the Multi vector retriever

from langchain_community.embeddings import OpenAIEmbeddings


parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=ollama_embeddings,
)

child_chunks_collection.reset_collection() #D

doc_byte_store = InMemoryByteStore() #E
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #F
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

In [29]:
import time
from ollama import ResponseError


def add_documents_in_batches(
    vectorstore,
    documents,
    batch_size=8,
    max_retries=3,
):
    """Embed and store documents in small, retryable batches."""

    for start in range(0, len(documents), batch_size):
        batch = documents[start:start + batch_size]

        for attempt in range(max_retries):
            try:
                vectorstore.add_documents(batch)
                break

            except ResponseError:
                if attempt == max_retries - 1:
                    raise

                delay = 2 ** attempt
                print(
                    f"Embedding batch failed; retrying in {delay} second(s)..."
                )
                time.sleep(delay)


# Ingesting the content into document and vector stores

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)  # A
    html_docs = html_loader.load()                  # B
    text_docs = html2text_transformer.transform_documents(
        html_docs
    )                                               # C

    coarse_chunks = parent_splitter.split_documents(
        text_docs
    )                                               # D

    coarse_chunks_ids = [
        str(uuid.uuid4()) for _ in coarse_chunks
    ]

    all_granular_chunks = []

    for i, coarse_chunk in enumerate(coarse_chunks):  # E
        coarse_chunk_id = coarse_chunks_ids[i]

        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]
        )                                             # F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id  # G

        all_granular_chunks.extend(granular_chunks)

    print(
        f"Ingesting {destination_url}: "
        f"{len(coarse_chunks)} parent chunks, "
        f"{len(all_granular_chunks)} child chunks"
    )

    add_documents_in_batches(
        vectorstore=multi_vector_retriever.vectorstore,
        documents=all_granular_chunks,
        batch_size=8,
    )                                                 # H

    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))
    )                                                 # I

# A Load one destination page.
# B Retrieve its HTML document.
# C Transform HTML into clean text.
# D Create coarse parent chunks.
# E Iterate over parent chunks.
# F Create granular child chunks.
# G Link every child chunk to its parent.
# H Embed and store child chunks in small batches.
# I Store parent chunks after all corresponding children succeed.

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.78it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall: 15 parent chunks, 133 child chunks


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.09it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall: 6 parent chunks, 45 child chunks


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.23it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England): 8 parent chunks, 57 child chunks


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.33it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford: 3 parent chunks, 24 child chunks


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.14it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest: 9 parent chunks, 70 child chunks


In [30]:
# Performing a search on granular information

retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")
len(retrieved_docs)

4

In [31]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Penzance', 'title': 'Penzance – Travel guide at Wikivoyage', 'language': 'en'}, page_content='First Kernow bus T1 runs every 30 min between Penzance and Truro (1 hr 45\nmin), via St Erth, Hayle, Camborne, and Redruth. Change at Truro for Newquay,\nSt Austell and Bodmin. Reaching Plymouth and Exeter by bus is not worth the\nbother, take the train.\n\n### By car\n\n[edit]\n\nPenzance is a 5- to 6-hour drive from London via M4, M5, and A30. It\'s a long\nway and at some point you\'ll need to refuel. Don\'t be paying motorway prices,\nthere\'s supermarket petrol at (amongst others) M5 jcn 28 (Cullompton Tesco),\nA30 Bodmin (Asda, Launceston Rd Bodmin) and A30 Penzance (Tesco).\n\n### By boat\n\n[edit]\n\nA ferry plies between Penzance and the Isles of Scilly, daily from mid-March\nto October. The ferry (Scillonian III[dead link]) leaves Penzance around 9AM\nto reach the main island of St Mary\'s at noon; it returns at 4:30PM for\

Note: **Same as Parent Document retriever, but more control and flexibility on how to link child to parent chunks.** [See](naive-rag-issues-and-solutions.md)

In [32]:
# Comparing with direct semantic search on child chunks

child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")
len(child_docs_only)

4

In [33]:
child_docs_only[0]

Document(id='b6d3bf68-1345-42d3-9c42-b67035247ec6', metadata={'source': 'https://en.wikivoyage.org/wiki/Penzance', 'language': 'en', 'doc_id': '2933dbda-e26c-47c3-bba2-02ffe62c006b', 'title': 'Penzance – Travel guide at Wikivoyage'}, page_content='These "A"-buses, operated by First Kernow, are blue open-top double-deckers in\nsummer. For bus travel plus rail, a good deal is the _Ride Cornwall Ranger_\n(adult £13) described above. For bus only, buy a _Day Rider_ for £12 (child\n£6) from the Bus Station or from the driver on boarding - contactless bank\ncards accepted. Bus drivers also issue Ride Cornwall Rangers, but only for\nfull price, go to the station for concessions.\n\n## See\n\n[edit]')